# Gold Layer — dim_customers
Build customer dimension from silver CRM + ERP tables.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session
from clickzetta.zettapark import functions as F
from clickzetta.zettapark.window import Window

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "gold",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## The Transformation Logic

In [ ]:
ci = session.table(f"silver.crm_customers")
ca = session.table(f"silver.erp_customers")
la = session.table(f"silver.erp_customer_location")

joined = (
    ci.join(ca, ci["customer_number"] == ca["customer_number"], "left")
      .join(la, ci["customer_number"] == la["customer_number"], "left")
)

# Select with explicit table refs to resolve column ambiguity after multi-join
df = joined.select(
    ci["customer_id"].alias("customer_id"),
    ci["customer_number"].alias("customer_number"),
    ci["first_name"].alias("first_name"),
    ci["last_name"].alias("last_name"),
    la["country"].alias("country"),
    ci["marital_status"].alias("marital_status"),
    F.when(ci["gender"] != "n/a", ci["gender"])
     .otherwise(F.coalesce(ca["gender"], F.lit("n/a"))).alias("gender"),
    ca["birth_date"].alias("birthdate"),
    ci["created_date"].alias("create_date"),
)

w = Window.order_by(F.col("customer_id"))
df = df.with_column("customer_key", F.row_number().over(w))
df = df.select(
    "customer_key", "customer_id", "customer_number",
    "first_name", "last_name", "country",
    "marital_status", "gender", "birthdate", "create_date",
)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Gold Table

In [ ]:
df.write.save_as_table(f"gold.dim_customers", mode="overwrite")
print("dim_customers OK")

## Verify

In [ ]:
session.table(f"gold.dim_customers").limit(5).show()